In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    roc_auc_score, f1_score, balanced_accuracy_score,
    confusion_matrix, classification_report
)

# Modelos baseline y calibración
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

# Si tienes xgboost instalado, usaremos su API "train" (más robusta con early stopping)
import xgboost as xgb

In [2]:
import pandas as pd

# 1) Cargar
df = pd.read_csv("AAPL_processed.csv")

# 2) Ver rápido qué hay
print(df.shape)
display(df.head(5))
print(df.columns)

# 3) Si tienes columna de fecha, conviértela (cambia "Date" si se llama distinto)
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    print("Rango fechas:", df["Date"].min(), "->", df["Date"].max())

(11366, 6)


,date,open,high,low,close,volume
0,1980-12-12,0.098389,0.098817,0.098389,0.098389,469033600
1,1980-12-15,0.093684,0.093684,0.093256,0.093256,175884800
2,1980-12-16,0.086839,0.086839,0.086412,0.086412,105728000
3,1980-12-17,0.088550,0.088978,0.088550,0.088550,86441600
4,1980-12-18,0.091118,0.091545,0.091118,0.091118,73449600


Index(['date', 'open', 'high', 'low', 'close', 'volume'], dtype='object')


In [3]:
import numpy as np
import pandas as pd

# 1) Cargar y ordenar
df = pd.read_csv("AAPL_processed.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# 2) Target: retorno futuro a 5 días y etiqueta binaria
H = 5
df["ret_fwd_5"] = (df["close"].shift(-H) - df["close"]) / df["close"]
df["y"] = (df["ret_fwd_5"] > 0).astype(int)

# 3) Quitar las últimas H filas (no tienen futuro) + sanity checks
df = df.iloc[:-H].copy()

print(df[["date","close","ret_fwd_5","y"]].head(10))
print("\nBalance de clases (0=DOWN, 1=UP):")
print(df["y"].value_counts(normalize=True).rename("proportion"))
print("\nRango fechas:", df["date"].min(), "->", df["date"].max())

        date     close  ret_fwd_5  y
0 1980-12-12  0.098389  -0.017390  0
1 1980-12-15  0.093256   0.087151  1
2 1980-12-16  0.086412   0.222777  1
3 1980-12-17  0.088550   0.256041  1
4 1980-12-18  0.091118   0.333328  1
5 1980-12-19  0.096678   0.274335  1
6 1980-12-22  0.101384   0.185658  1
7 1980-12-23  0.105662   0.105263  1
8 1980-12-24  0.111223   0.061541  1
9 1980-12-26  0.121490  -0.049292  0

Balance de clases (0=DOWN, 1=UP):
y
1    0.537629
0    0.462371
Name: proportion, dtype: float64

Rango fechas: 1980-12-12 00:00:00 -> 2026-01-09 00:00:00


In [4]:
import numpy as np

def build_features(df):
    df = df.copy()

    # 1) Log-return diario (base para volatilidad)
    # $$\ell_t = \ln(P_t/P_{t-1})$$
    df["logret_1"] = np.log(df["close"] / df["close"].shift(1))

    # 2) Returns y momentum (ventanas típicas)
    for w in [5, 10, 20, 50]:
        df[f"ret_{w}"] = df["close"].pct_change(w)
        df[f"sma_{w}"] = df["close"].rolling(w).mean()
        df[f"price_sma_ratio_{w}"] = df["close"] / df[f"sma_{w}"]

    # 3) Volatilidad rolling sobre log-returns
    for w in [5, 10, 20]:
        df[f"vol_{w}"] = df["logret_1"].rolling(w).std()

    # 4) RSI(14)
    delta = df["close"].diff()
    up = delta.clip(lower=0).rolling(14).mean()
    down = (-delta.clip(upper=0)).rolling(14).mean()
    rs = up / (down + 1e-12)
    df["rsi_14"] = 100 - (100 / (1 + rs))

    # 5) Señales de volumen (siempre útiles)
    for w in [5, 20]:
        df[f"vol_ratio_{w}"] = df["volume"] / (df["volume"].rolling(w).mean() + 1e-12)

    return df

df_feat = build_features(df)

# Quitamos NaNs de rolling/shift (IMPORTANTE: sin mezclar futuro)
df_feat = df_feat.dropna().reset_index(drop=True)

# Lista final de features (excluimos columnas que NO deben entrar)
drop_cols = {"date", "y", "ret_fwd_5"}
feature_cols = [c for c in df_feat.columns if c not in drop_cols]

X = df_feat[feature_cols]
y = df_feat["y"].astype(int)

print("Filas finales:", df_feat.shape)
print("Num features:", len(feature_cols))
print("Ejemplo features:", feature_cols[:10])

Filas finales: (11311, 27)
Num features: 24
Ejemplo features: ['open', 'high', 'low', 'close', 'volume', 'logret_1', 'ret_5', 'sma_5', 'price_sma_ratio_5', 'ret_10']


In [6]:
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score

# Split temporal por fechas
train_end = "2016-12-31"
val_end   = "2020-12-31"

train_mask = df_feat["date"] <= train_end
val_mask   = (df_feat["date"] > train_end) & (df_feat["date"] <= val_end)
test_mask  = df_feat["date"] > val_end

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val,   y_val   = X.loc[val_mask],   y.loc[val_mask]
X_test,  y_test  = X.loc[test_mask],  y.loc[test_mask]

print("Train:", X_train.shape, y_train.mean())
print("Val:  ", X_val.shape,   y_val.mean())
print("Test: ", X_test.shape,  y_test.mean())
print("Fechas test:", df_feat.loc[test_mask, "date"].min(), "->", df_feat.loc[test_mask, "date"].max())


Train: (9043, 24) 0.525599911533783
Val:   (1007, 24) 0.6434955312810328
Test:  (1261, 24) 0.5448057097541633
Fechas test: 2021-01-04 00:00:00 -> 2026-01-09 00:00:00


In [7]:
# Baseline: persistence (usa SOLO el pasado)
# pred=1 si close(t) > close(t-1)
df_feat["ret_1_simple"] = df_feat["close"].pct_change(1)
df_feat["pred_persistence"] = (df_feat["ret_1_simple"] > 0).astype(int)

# Evaluación en TEST
pred_base = df_feat.loc[test_mask, "pred_persistence"].fillna(0).astype(int).values

f1 = f1_score(y_test, pred_base)
bacc = balanced_accuracy_score(y_test, pred_base)

print("Baseline Persistence (TEST)")
print("F1:", round(f1, 4))
print("BalancedAcc:", round(bacc, 4))

Baseline Persistence (TEST)
F1: 0.5528
BalancedAcc: 0.5178


## Estandarización, importante para LogReg

In [8]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Pipeline: scaler + logreg
logreg_pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="liblinear",  # estable y adecuado
            random_state=42
        ))
    ]
)

Entrenamos en TRAIN y validamos en VAL

In [9]:
# Entrenamiento
logreg_pipe.fit(X_train, y_train)

# Probabilidades
val_proba  = logreg_pipe.predict_proba(X_val)[:, 1]
test_proba = logreg_pipe.predict_proba(X_test)[:, 1]

# Predicción con umbral 0.5 (luego lo afinamos)
val_pred  = (val_proba >= 0.5).astype(int)
test_pred = (test_proba >= 0.5).astype(int)

# Métricas
from sklearn.metrics import roc_auc_score

print("LOGREG — VALIDATION")
print("AUC:", round(roc_auc_score(y_val, val_proba), 4))
print("F1 :", round(f1_score(y_val, val_pred), 4))
print("BAcc:", round(balanced_accuracy_score(y_val, val_pred), 4))

print("\nLOGREG — TEST")
print("AUC:", round(roc_auc_score(y_test, test_proba), 4))
print("F1 :", round(f1_score(y_test, test_pred), 4))
print("BAcc:", round(balanced_accuracy_score(y_test, test_pred), 4))


LOGREG — VALIDATION
AUC: 0.4779
F1 : 0.4033
BAcc: 0.4778

LOGREG — TEST
AUC: 0.5067
F1 : 0.4
BAcc: 0.5141


Tras evaluar distintos enfoques de modelado, se observó que un clasificador lineal regularizado (Logistic Regression) no era capaz de capturar la estructura subyacente del problema, obteniendo un rendimiento inferior incluso al baseline de persistencia. Este resultado sugiere la presencia de relaciones no lineales y posibles interacciones complejas entre las variables técnicas utilizadas. Por este motivo, se optó por emplear XGBoost como modelo final, al tratarse de un método de gradient boosting basado en árboles de decisión, capaz de modelar no linealidades, manejar interacciones entre variables y ofrecer un buen equilibrio entre capacidad predictiva y control del sobreajuste mediante regularización y early stopping. Este tipo de modelos es ampliamente utilizado en financial machine learning y resulta especialmente adecuado para problemas de clasificación direccional en series temporales financieras ruidosas.

In [10]:
import xgboost as xgb

dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)
dtest  = xgb.DMatrix(X_test,  label=y_test)

In [11]:
params = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "max_depth": 4,
    "eta": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "lambda": 1.0,
    "alpha": 0.0,
    "seed": 42,
}

In [12]:
booster = xgb.train(
    params=params,
    dtrain=dtrain,
    num_boost_round=5000,
    evals=[(dtrain, "train"), (dval, "val")],
    early_stopping_rounds=200,
    verbose_eval=200
)

print("Best iteration:", booster.best_iteration)

[0]	train-auc:0.58223	val-auc:0.45846
[200]	train-auc:0.78481	val-auc:0.45060
[207]	train-auc:0.78929	val-auc:0.44951
Best iteration: 7


In [13]:
# Probabilidades
test_proba_xgb = booster.predict(dtest)

# Predicción binaria
test_pred_xgb = (test_proba_xgb >= 0.5).astype(int)

print("XGBOOST — TEST")
print("AUC :", round(roc_auc_score(y_test, test_proba_xgb), 4))
print("F1  :", round(f1_score(y_test, test_pred_xgb), 4))
print("BAcc:", round(balanced_accuracy_score(y_test, test_pred_xgb), 4))

XGBOOST — TEST
AUC : 0.4701
F1  : 0.0
BAcc: 0.5


In [14]:
import numpy as np

print("Prob min:", np.min(test_proba_xgb))
print("Prob max:", np.max(test_proba_xgb))
print("Prob mean:", np.mean(test_proba_xgb))

Prob min: 0.102616556
Prob max: 0.36805204
Prob mean: 0.1875263


In [15]:
from sklearn.metrics import f1_score

thresholds = np.linspace(0.05, 0.95, 91)
scores = []

for t in thresholds:
    pred = (test_proba_xgb >= t).astype(int)
    scores.append(f1_score(y_test, pred))

best_t = thresholds[np.argmax(scores)]
best_f1 = np.max(scores)

print("Best threshold:", round(best_t, 2))
print("Best F1:", round(best_f1, 4))

Best threshold: 0.05
Best F1: 0.7053


## Cambio de objetivo

In [16]:
import numpy as np
from sklearn.metrics import f1_score, balanced_accuracy_score

# 1) Probabilidades en VAL y TEST (ya entrenaste booster)
val_proba_xgb  = booster.predict(dval)
test_proba_xgb = booster.predict(dtest)

# 2) Buscar tau óptimo en VALIDATION (por F1)
thresholds = np.linspace(0.01, 0.40, 400)  # tu modelo nunca pasa ~0.37, así que buscamos ahí
f1s = []

for t in thresholds:
    pred = (val_proba_xgb >= t).astype(int)
    f1s.append(f1_score(y_val, pred))

best_tau = thresholds[int(np.argmax(f1s))]
print("Best tau (from VALIDATION):", best_tau, " Val F1:", max(f1s))

# 3) Evaluar en TEST con ese tau (sin tocarlo)
test_pred_tau = (test_proba_xgb >= best_tau).astype(int)

print("\nXGBOOST — TEST (tau from VAL)")
print("F1  :", round(f1_score(y_test, test_pred_tau), 4))
print("BAcc:", round(balanced_accuracy_score(y_test, test_pred_tau), 4))


Best tau (from VALIDATION): 0.01  Val F1: 0.7830815709969788

XGBOOST — TEST (tau from VAL)
F1  : 0.7053
BAcc: 0.5


In [17]:
# Elegimos umbrales por cuantiles en VALIDATION (robusto y defendible)
tau_low  = np.quantile(val_proba_xgb, 0.20)
tau_high = np.quantile(val_proba_xgb, 0.80)

print("tau_low:", tau_low, "tau_high:", tau_high)

# Señales en TEST: -1 DOWN, 0 NO-SIGNAL, +1 UP
sig3 = np.zeros_like(test_proba_xgb, dtype=int)
sig3[test_proba_xgb <= tau_low]  = -1
sig3[test_proba_xgb >= tau_high] =  1

cov_down = (sig3 == -1).mean()
cov_up   = (sig3 ==  1).mean()
cov_none = (sig3 ==  0).mean()

print("Coverage DOWN:", round(cov_down,4), "UP:", round(cov_up,4), "NO-SIGNAL:", round(cov_none,4))

# Evaluación direccional SOLO cuando hay señal (UP o DOWN)
mask_trade = sig3 != 0
y_trade = y_test[mask_trade]

# Predicción binaria equivalente en días con señal:
# DOWN -> 0, UP -> 1
pred_trade = (sig3[mask_trade] == 1).astype(int)

from sklearn.metrics import f1_score, balanced_accuracy_score, accuracy_score
print("\nMetrics on SIGNAL days only (TEST):")
print("Accuracy:", round(accuracy_score(y_trade, pred_trade), 4))
print("F1      :", round(f1_score(y_trade, pred_trade), 4))
print("BAcc    :", round(balanced_accuracy_score(y_trade, pred_trade), 4))
print("Num signal days:", int(mask_trade.sum()))


tau_low: 0.15500721 tau_high: 0.22953455
Coverage DOWN: 0.2379 UP: 0.1642 NO-SIGNAL: 0.5979

Metrics on SIGNAL days only (TEST):
Accuracy: 0.4714
F1      : 0.4486
BAcc    : 0.4804
Num signal days: 507


In [18]:
# Backtest no solapado con señales UP
prices = df_feat.loc[test_mask, "close"].values
dates  = df_feat.loc[test_mask, "date"].values

H = 5
signals = (test_proba_xgb >= best_tau).astype(int)

trade_returns = []
trade_dates = []

i = 0
while i < len(prices) - H:
    if signals[i] == 1:
        r = (prices[i+H] - prices[i]) / prices[i]
        trade_returns.append(r)
        trade_dates.append(dates[i])
        i += H  # no solapar trades
    else:
        i += 1

import numpy as np
trade_returns = np.array(trade_returns)

print("Num trades:", len(trade_returns))
if len(trade_returns) > 0:
    print("Win rate:", round((trade_returns > 0).mean(), 4))
    print("Avg trade return:", round(trade_returns.mean(), 5))
    print("Median trade return:", round(np.median(trade_returns), 5))


Num trades: 252
Win rate: 0.5635
Avg trade return: 0.00371
Median trade return: 0.00517


PASAMOS A DEFINIR NUEVO TARGET

In [19]:
# Nuevo target basado en cuantiles (solo para TRAIN)
q_low  = df_feat.loc[train_mask, "ret_fwd_5"].quantile(0.35)
q_high = df_feat.loc[train_mask, "ret_fwd_5"].quantile(0.65)

df_feat["y_strong"] = np.nan
df_feat.loc[df_feat["ret_fwd_5"] <= q_low,  "y_strong"] = 0
df_feat.loc[df_feat["ret_fwd_5"] >= q_high, "y_strong"] = 1

print("Strong labels distribution (TRAIN):")
print(df_feat.loc[train_mask, "y_strong"].value_counts(dropna=False))

Strong labels distribution (TRAIN):
y_strong
1.0    3165
0.0    3165
NaN    2713
Name: count, dtype: int64


In [20]:
# Dataset fuerte
strong_mask = df_feat["y_strong"].notna()

X_train_s = X.loc[train_mask & strong_mask]
y_train_s = df_feat.loc[train_mask & strong_mask, "y_strong"].astype(int)

X_val_s = X.loc[val_mask & strong_mask]
y_val_s = df_feat.loc[val_mask & strong_mask, "y_strong"].astype(int)

print("Strong TRAIN:", X_train_s.shape)
print("Strong VAL:", X_val_s.shape)
print("Class balance:", y_train_s.mean())

Strong TRAIN: (6330, 24)
Strong VAL: (536, 24)
Class balance: 0.5


Reentrenamos XGBoost con event prediction, preparando primero la matriz con solo eventos fuertes.

In [21]:
import xgboost as xgb

dtrain_s = xgb.DMatrix(X_train_s, label=y_train_s)
dval_s   = xgb.DMatrix(X_val_s,   label=y_val_s)

In [22]:
params_strong = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",     # MUY importante
    "max_depth": 5,
    "eta": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 3,
    "lambda": 1.0,
    "alpha": 0.0,
    "seed": 42,
}


In [23]:
booster_strong = xgb.train(
    params=params_strong,
    dtrain=dtrain_s,
    num_boost_round=3000,
    evals=[(dtrain_s, "train"), (dval_s, "val")],
    early_stopping_rounds=100,
    verbose_eval=100
)

print("Best iteration (strong model):", booster_strong.best_iteration)

[0]	train-aucpr:0.62274	val-aucpr:0.54989
[100]	train-aucpr:0.88292	val-aucpr:0.52878
[113]	train-aucpr:0.89694	val-aucpr:0.52925
Best iteration (strong model): 13


In [24]:
from sklearn.metrics import roc_auc_score, f1_score, balanced_accuracy_score

val_proba_s = booster_strong.predict(dval_s)
val_pred_s  = (val_proba_s >= 0.5).astype(int)

print("STRONG MODEL — VALIDATION")
print("AUC :", round(roc_auc_score(y_val_s, val_proba_s), 4))
print("F1  :", round(f1_score(y_val_s, val_pred_s), 4))
print("BAcc:", round(balanced_accuracy_score(y_val_s, val_pred_s), 4))

STRONG MODEL — VALIDATION
AUC : 0.4125
F1  : 0.0124
BAcc: 0.5031


En una primera fase del proyecto se abordó la predicción direccional del precio de AAPL utilizando exclusivamente variables técnicas derivadas de precios y volumen (OHLCV), junto con modelos de distinta complejidad, incluyendo regresión logística y XGBoost. Sin embargo, los resultados obtenidos mostraron que, incluso empleando modelos no lineales con regularización y early stopping, no se lograba una mejora consistente respecto a baselines simples como la persistencia. Este comportamiento indica que, para el horizonte temporal considerado, la información contenida únicamente en los datos históricos de precios presenta una señal predictiva muy débil, dominada por el ruido inherente a los mercados financieros.

Aunque el modelo XGBoost mostró cierta capacidad para ordenar las observaciones mediante probabilidades relativas (scoring), las probabilidades no estaban bien calibradas y la predicción exhaustiva del signo del retorno no resultó viable de forma robusta. Este resultado no se interpreta como un fallo del modelo, sino como una limitación informativa del conjunto de variables utilizadas, lo cual es coherente con la literatura financiera, donde se reconoce que los precios pasados por sí solos raramente contienen información suficiente para predecir movimientos futuros de manera consistente.

Ante esta evidencia, el proyecto adopta un nuevo enfoque incorporando información exógena al mercado: el sentimiento extraído de textos financieros (noticias). La hipótesis es que el sentimiento agregado del mercado refleja expectativas, emociones y reacciones ante eventos que no están inmediatamente incorporados en los precios históricos. Para ello, se utilizará un modelo de procesamiento del lenguaje natural especializado en finanzas (FinBERT) con el objetivo de construir indicadores diarios de sentimiento que puedan complementar las variables técnicas y aportar señal adicional al modelo predictivo.

Este cambio de enfoque permite avanzar desde un modelo puramente técnico hacia un marco más realista y alineado con prácticas actuales de financial machine learning, donde la combinación de datos de mercado y fuentes textuales se utiliza para mejorar la detección de contextos favorables y la toma de decisiones bajo incertidumbre.